In [1]:
from chemprop.nn import BondMessagePassing
import torch
import ray

from data.loaders import load_dataset
from data import SupportedDatasets
from data.preprocessing import preprocess_ray

from train import generate_repeated_5xn_splits, train_and_evaluate_split, TrainConfig
from config import SplitType

In [2]:
%load_ext autoreload
%autoreload 2

In [42]:
from chemprop.conf import DEFAULT_ATOM_FDIM, DEFAULT_BOND_FDIM, DEFAULT_HIDDEN_DIM
from chemprop.data import BatchMolGraph
from chemprop.nn import Activation, GraphTransform, ScaleTransform
from torch import Tensor
from torch.nn.modules import Module


class ResidualFFN(torch.nn.Module):
    def __init__(self, dims: int) -> None:
        super().__init__()

        self.norm = torch.nn.LayerNorm(dims)
        self.non_linear = torch.nn.Sequential(*[
            torch.nn.Linear(dims, dims),
            torch.nn.PReLU(),
            torch.nn.Linear(dims, dims),
        ])

    def forward(self, inp, res):
        return self.non_linear(self.norm(inp)) + res


class ModdedBondMessagePassing(BondMessagePassing):
    def __init__(
        self,
        d_v: int = DEFAULT_ATOM_FDIM,
        d_e: int = DEFAULT_BOND_FDIM,
        d_h: int = DEFAULT_HIDDEN_DIM,
        bias: bool = False,
        depth: int = 3,
        dropout: float = 0,
        activation: str | Module | Activation = Activation.RELU,
        undirected: bool = False,
        d_vd: int | None = None,
        V_d_transform: ScaleTransform | None = None,
        graph_transform: GraphTransform | None = None,
    ):
        super().__init__(
            d_v,
            d_e,
            d_h,
            bias,
            depth,
            dropout,
            activation,
            undirected,
            d_vd,
            V_d_transform,
            graph_transform,
        )

        # self.layer_ffn = torch.nn.ModuleList([ResidualFFN(d_h) for _ in range(depth)])
        self.layer_ffn = ResidualFFN(d_h)

    
    def update(self, M_t, H_0, H_prev, t):
        """Calcualte the updated hidden for each edge"""
        H_t = self.W_h(M_t)
        H_t = self.tau(H_0 + H_t)
        H_t = self.dropout(H_t)

        # H_t = self.layer_ffn[t](H_t) + H_prev
        H_t = self.layer_ffn(H_t, H_prev)
        return H_t
    

    def forward(self, bmg: BatchMolGraph, V_d: Tensor | None = None) -> Tensor:
        bmg = self.graph_transform(bmg)
        H_0 = self.initialize(bmg)

        H = self.tau(H_0)
        for t in range(1, self.depth):
            if self.undirected:
                H = (H + H[bmg.rev_edge_index]) / 2

            M = self.message(H, bmg)
            H = self.update(M, H_0, H, t-1)

        index_torch = bmg.edge_index[1].unsqueeze(1).repeat(1, H.shape[1])
        M = torch.zeros(len(bmg.V), H.shape[1], dtype=H.dtype, device=H.device).scatter_reduce_(
            0, index_torch, H, reduce="sum", include_self=False
        )
        return self.finalize(M, bmg.V, V_d)

In [25]:
df, df_classification_threshold = load_dataset(SupportedDatasets.SINGLE_TARGET_TBA)
df = preprocess_ray(df)
ray.shutdown()

splits = generate_repeated_5xn_splits(df, n=2, split_type=SplitType.SCAFFOLD, random_state=42)
# _ = next(splits)
_, (train_df, val_df, test_df) = next(splits)

/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
2026-05-17 15:38:37,887	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/ray/data/dataset.py:1589: UserWarning: Use 'expr' instead of 'fn' when possible for performant filters.
  warnings.warn(
2026-05-17 15:38:41,510	INFO logging.py:392 -- Registered dataset logger for dataset dataset_2_0
2026-05-17 15:38:41,516	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-05-17_15-38-32_533343_287812/logs/ray-data
2026-05-17 15:38:41,518	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> TaskPoolMapOperator[Map(<lambda>)->Fil

In [26]:
from models import baseline, baseline_modded

res_baseline = train_and_evaluate_split(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    df_classification_threshold=df_classification_threshold,
    model_module=baseline,
    model_config=baseline.BaselineConfig(mp_depth=2),
    train_config=TrainConfig(batch_size=32, max_epochs=30),
    # modded_bond_message_passing=ModdedBondMessagePassing
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (8) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ NormAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │  180 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 408 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 408 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 28                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:365: Skipping 'metrics' parameter because it is not possible to safely dump to YAML.


...

Metric val_loss improved by 0.055 >= min_delta = 0.0. New best score: 0.588


Metric val_loss improved by 0.003 >= min_delta = 0.0. New best score: 0.585


Metric val_loss improved by 0.021 >= min_delta = 0.0. New best score: 0.564


Metric val_loss improved by 0.011 >= min_delta = 0.0. New best score: 0.552


`Trainer.fit` stopped: `max_epochs=30` reached.


In [43]:
res_modded = train_and_evaluate_split(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    df_classification_threshold=df_classification_threshold,
    model_module=baseline_modded,
    model_config=baseline.BaselineConfig(mp_depth=8),
    train_config=TrainConfig(batch_size=32, max_epochs=30),
    modded_bond_message_passing=ModdedBondMessagePassing
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


{'binary_threshold': GT(th=0.5), 'modded_bond_message_passing': <class '__main__.ModdedBondMessagePassing'>}


/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (8) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ ModdedBondMessagePassing │  408 K │ train │     0 │
│ 1 │ agg             │ NormAggregation          │      0 │ train │     0 │
│ 2 │ bn              │ Identity                 │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN  │  180 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                 │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList               │      0 │ train │     0 │
└───┴─────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 589 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 589 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Metric val_loss improved by 0.016 >= min_delta = 0.0. New best score: 0.591


Metric val_loss improved by 0.011 >= min_delta = 0.0. New best score: 0.579


Metric val_loss improved by 0.009 >= min_delta = 0.0. New best score: 0.571


Metric val_loss improved by 0.004 >= min_delta = 0.0. New best score: 0.567


`Trainer.fit` stopped: `max_epochs=30` reached.


In [28]:
res_baseline

{'roc_auc': 0.7474747474747474, 'average_precision': 0.7487099471515629}

In [35]:
res_modded

{'roc_auc': 0.768939393939394, 'average_precision': 0.7911206737717361}

In [44]:
res_modded

{'roc_auc': 0.7696969696969698, 'average_precision': 0.7713961638344587}